In [9]:
!pip install transformers datasets torch torchcodec torchaudio speechbrain

In [10]:
import torch
from transformers import SpeechT5ForTextToSpeech, SpeechT5Processor, SpeechT5HifiGan
from datasets import load_dataset
from IPython.display import Audio, display


In [12]:

def load_speech_model(checkpoint="bilalfaye/speecht5_tts-wolof-v0.2", vocoder_checkpoint="microsoft/speecht5_hifigan"):
    """ Load the SpeechT5 model, processor, and vocoder for text-to-speech. """

    device = torch.device("cpu")

    processor = SpeechT5Processor.from_pretrained(checkpoint)
    model = SpeechT5ForTextToSpeech.from_pretrained(checkpoint).to(device)
    vocoder = SpeechT5HifiGan.from_pretrained(vocoder_checkpoint).to(device)

    return processor, model, vocoder, device


In [ ]:
processor, model, vocoder, device = load_speech_model()

# Temporarily use a random speaker embedding as Matthijs/cmu-arctic-xvectors
# is no longer supported and speechbrain/speaker-embeddings-cmu-arctic-xvectors was not found.
# For a specific speaker, you would typically generate an embedding from an audio file.
speaker_embedding = torch.randn(1, 512).to(device) # Generate a random 512-dimension embedding and move to device




ImportError: 
SpeechT5Tokenizer requires the SentencePiece library but it was not found in your environment. Check out the instructions on the
installation page of its repo: https://github.com/google/sentencepiece#installation and follow the ones
that match your environment. Please note that you may need to restart your runtime after installation.


In [ ]:
def generate_speech_from_text(input_text, speaker_embedding=speaker_embedding, processor=processor, model=model, vocoder=vocoder, use_ssml=False):
    """ Generates speech from input text using SpeechT5 and HiFi-GAN vocoder.
        If use_ssml is True, input_text is treated as SSML.
        Example SSML for pauses: "<break time=\"500ms\"/>"
    """
    # SpeechT5Processor tokenizes the raw text, including SSML tags. The model then interprets them.
    inputs = processor(text=input_text, return_tensors="pt", padding=True, truncation=True, max_length=model.config.max_text_positions)
    inputs = {key: value.to(model.device) for key, value in inputs.items()}

    speech = model.generate(
        inputs["input_ids"],
        speaker_embeddings=speaker_embedding.to(model.device),
        vocoder=vocoder,
        num_beams=7,
        temperature=0.6,
        no_repeat_ngram_size=3,
        repetition_penalty=1.5,
    )

    speech = speech.detach().cpu().numpy()
    display(Audio(speech, rate=16000))


In [ ]:

# Example usage French (plain text)
text_fr = "Bonjour, bienvenue dans le modèle de synthèse vocale Wolof et Français."
generate_speech_from_text(text_fr)

# Example usage Wolof (plain text)
text_wolof = "ñu ne ñoom ñooy nattukaay satélite yi"
generate_speech_from_text(text_wolof)

# Example usage French with SSML for pauses
ssml_text_fr = "Bonjour,<break time=\"750ms\"/> bienvenue dans le modèle de synthèse vocale Wolof et Français.<break time=\"1s\"/> J'espère que cela vous convient mieux."
generate_speech_from_text(ssml_text_fr, use_ssml=True)

# Example usage Wolof with SSML for pauses
ssml_text_wolof = "ñu ne ñoom ñooy nattukaay satélite yi.<break time=\"750ms\"/> Foo deuk la beug xam?"
generate_speech_from_text(ssml_text_wolof, use_ssml=True)

In [ ]:
import torchaudio
from speechbrain.pretrained import EncoderClassifier

# Charger le modèle d'encodage de locuteur
# Ce modèle extrait les x-vectors, qui sont des types d'embeddings de locuteur.
# Le modèle est téléchargé la première fois, cela peut prendre un certain temps.
speaker_encoder = EncoderClassifier.from_hparams(source="speechbrain/spkrec-xvect-voxceleb", savedir="pretrained_models/speaker_encoder")

audio_file_path = "./audio.wav"  # <--- REMPLACEZ CE CHEMIN !

# Charger le fichier audio
signal, fs = torchaudio.load(audio_file_path)

# Normaliser la fréquence d'échantillonnage si nécessaire (SpeechT5 attend 16kHz)
if fs != 16000:
    resampler = torchaudio.transforms.Resample(orig_freq=fs, new_freq=16000)
    signal = resampler(signal)

# Assurez-vous que le signal est mono et a une dimension de batch
# torchaudio.load retourne (num_channels, num_samples)
# Nous voulons (batch_size, num_samples) pour speechbrain.
# Nous prenons le premier canal et ajoutons une dimension de batch.
signal = signal[0].unsqueeze(0) # Prend le premier canal et ajoute la dimension de batch

# Extraire le "speaker embedding"
with torch.no_grad():
    speaker_embedding_new = speaker_encoder.encode_batch(signal.to(device))

print(f"Speaker embedding généré avec succès de la forme : {speaker_embedding_new.shape}")

# Mettre à jour la variable globale speaker_embedding pour l'utiliser dans la fonction generate_speech_from_text
speaker_embedding = speaker_embedding_new

# Exemple usage French avec le nouvel embedding
text = "Bonjour, bienvenue dans le modèle de synthèse vocale Wolof et Français."
generate_speech_from_text(text)

